In [1]:
####################################################################################################
### NOTEBOOK: 113_Final_Precision_Strike.ipynb
####################################################################################################

import pandas as pd
import numpy as np
import os
from tqdm import tqdm

# 1. НАСТРОЙКИ
SUBMISSIONS_DIR = "submissions"
TARGETS = {
    "278400": 278400.0,
    "278600": 278600.0
}

PATHS = {
    "y_30m":   f"{SUBMISSIONS_DIR}/081_30min_BEST_QUALITY_RAW.csv",
    "y_bq1h":  f"{SUBMISSIONS_DIR}/autogluon_best_quality_sub.csv",
    "y_spec":  f"{SUBMISSIONS_DIR}/autogluon_special_no_k.csv",
    "y_pulse": f"{SUBMISSIONS_DIR}/tf_global_pulse_sub.csv",
    "y_ridge": f"{SUBMISSIONS_DIR}/105_Ridge_Legion_RAW.csv"
}

# 2. ЗАГРУЗКА КОМПОНЕНТОВ
print("Loading Exodia components...")
dfs = {name: pd.read_csv(path).sort_values('id').reset_index(drop=True) for name, path in PATHS.items()}
base_ids = dfs["y_30m"]['id']
test_meta = pd.read_parquet('data/test_solo_track.parquet')[['id', 'route_id', 'timestamp']]
test_meta['timestamp'] = pd.to_datetime(test_meta['timestamp'])

# 3. БЛЕНДИНГ (DISSENSUS LOGIC)
print("Calculating Dynamic Dissensus Weights...")
all_preds = np.column_stack([dfs[n]['y_pred'].values for n in ["y_30m", "y_bq1h", "y_spec", "y_pulse", "y_ridge"]])

final_y_raw = []
for i in range(len(all_preds)):
    row = all_preds[i]
    cv = np.std(row) / (np.mean(row) + 1e-6)
    
    # Базовые веса
    w = np.array([0.30, 0.20, 0.15, 0.20, 0.15])
    # Если модели спорят (CV > 0.3), доверяем Ridge Legion
    if cv > 0.3:
        w = np.array([0.20, 0.10, 0.10, 0.20, 0.40])
    
    w /= w.sum()
    final_y_raw.append(np.dot(row, w))

raw_blend_df = pd.DataFrame({'id': base_ids, 'route_id': test_meta['route_id'], 'y_pred_raw': final_y_raw})

# 4. РАСЧЕТ НАСЛЕДСТВА (VISE)
print("Calculating Physical Heritage (10:30 AM)...")
train = pd.read_parquet('data/train_solo_track.parquet')
def get_heritage_30m(target_series):
    T = target_series.values.astype(np.float64)
    n = len(T)
    v30 = np.full(n, np.nan)
    zero_idx = np.where(T == 0)[0]
    if len(zero_idx) > 0:
        for idx in zero_idx:
            v30[idx] = 0
            if idx > 0: v30[idx-1] = 0
        for _ in range(3):
            for i in range(1, n):
                if np.isnan(v30[i]) and not np.isnan(v30[i-1]): v30[i] = max(0, T[i] - v30[i-1])
            for i in range(n-1, 0, -1):
                if np.isnan(v30[i-1]) and not np.isnan(v30[i]): v30[i-1] = max(0, T[i] - v30[i])
    return np.nan_to_num(v30, nan=T[-1]/2)[-1]

heritage_dict = {rid: get_heritage_30m(train[train['route_id'] == rid].sort_values('timestamp')['target_1h']) for rid in range(1000)}

# 5. ЦИКЛ ГЕНЕРАЦИИ ФИНАЛЬНЫХ ФАЙЛОВ
current_raw_mean = raw_blend_df['y_pred_raw'].mean()

for name, target_val in TARGETS.items():
    print(f"\n--- Processing Target: {target_val} ---")
    
    # А. Масштабирование
    scaling_factor = target_val / current_raw_mean
    sub = raw_blend_df.copy()
    sub['y_pred_calibrated'] = sub['y_pred_raw'] * scaling_factor
    
    # Б. Применение Тисков (Vise Fix)
    sub = sub.merge(test_meta[['id', 'timestamp']], on='id')
    sub['v1030'] = sub['route_id'].map(heritage_dict)
    mask_1100 = (sub['timestamp'].dt.hour == 11) & (sub['timestamp'].dt.minute == 0)
    
    sub['y_pred'] = sub['y_pred_calibrated']
    sub.loc[mask_1100, 'y_pred'] = np.maximum(sub.loc[mask_1100, 'y_pred'], sub.loc[mask_1100, 'v1030'])
    
    # В. Математическая тюрьма (Triple Window Fix)
    fix_count = 0
    for rid in range(1000):
        idx = sub[sub['route_id'] == rid].index
        preds = sub.loc[idx, 'y_pred'].values
        for t in range(0, 6):
            if preds[t] + preds[t+2] < preds[t+1]:
                preds[t+1] = preds[t] + preds[t+2]
                fix_count += 1
        sub.loc[idx, 'y_pred'] = preds
    
    # Г. Сохранение
    out_path = f"{SUBMISSIONS_DIR}/113_FINAL_DISSENSUS_{name}.csv"
    sub[['id', 'y_pred']].sort_values('id').to_csv(out_path, index=False)
    
    print(f"✅ Saved: {out_path}")
    print(f"   Scaling Factor: {scaling_factor:.4f}")
    print(f"   Triple Window fixes: {fix_count}")
    print(f"   Final Mean: {sub['y_pred'].mean():.2f}")

print("\n🚀 ВСЕ ФИНАЛЬНЫЕ САБМИТЫ ГОТОВЫ. УДАЧИ НА ПРИВАТЕ!")

Loading Exodia components...
Calculating Dynamic Dissensus Weights...
Calculating Physical Heritage (10:30 AM)...

--- Processing Target: 278400.0 ---
✅ Saved: submissions/113_FINAL_DISSENSUS_278400.csv
   Scaling Factor: 1.0592
   Triple Window fixes: 0
   Final Mean: 278402.85

--- Processing Target: 278600.0 ---
✅ Saved: submissions/113_FINAL_DISSENSUS_278600.csv
   Scaling Factor: 1.0600
   Triple Window fixes: 0
   Final Mean: 278602.82

🚀 ВСЕ ФИНАЛЬНЫЕ САБМИТЫ ГОТОВЫ. УДАЧИ НА ПРИВАТЕ!
